# Model Training Notebook

This notebook is used to launch model training process and save results.

In [ ]:
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader
from model.dataset import SuperResDataset
from model.lit_residual_upscaler import LitSuperResNet
from model.lit_image_logger import ImageLoggerCallback

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# .env setup

This notebook uses .env file to get data paths from. This file is specific to the local machine and should be created manually.

**Make sure .env file is presented in the project root and following entries are added:**
- TRAIN_DATA_PATH - path to folder containing training high-res images
- VAL_DATA_PATH - path to folder containing validation images
- TB_LOGGING_DATA_PATH - path to folder containing images for Tensorboard logging (it is useful to compare training dynamics of multiple runs on Tensorboard)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from skimage.io import imread
import os

TRAIN_PATH = os.getenv("TRAIN_DATA_PATH")
VAL_PATH = os.getenv("VAL_DATA_PATH")

HIRES_PATCH_SIZE = 128
DOWNSCALED_SIZE = (5616 // 4, 3744 // 4) # 1/4-res of Canon 5dII frame was used in training to reduce camera matrix noise

train_dataset = SuperResDataset(
    TRAIN_PATH, 
    patch_size=HIRES_PATCH_SIZE,
    downscale=DOWNSCALED_SIZE,
)
val_dataset = SuperResDataset(
    VAL_PATH, 
    patch_size=HIRES_PATCH_SIZE,
    downscale=DOWNSCALED_SIZE,
)

assert len(train_dataset) > 0, "No training data found"
assert len(val_dataset) > 0, "No validation data found"
print(f'Loaded {len(train_dataset)} train images, {len(val_dataset)} val images')

Here we define a name for current model version and it's hyperparameters. Resume from checkpoint is also available here if needed.

**Make sure that each training process is followed by an unique comment value to distinguish multiple models later.**

In [ ]:
COMMENT = 'v3.2_per_channel_post'

model = LitSuperResNet(
    lr=1e-4, 
    num_blocks=14, 
    block_length=2, 
    channels=128,
    lambda_y=1,
    lambda_cbcr=0.25,
    lambda_grad=0.01,
    lambda_laplasian=0,
    lambda_tiny=0,
    loss_warmup_epochs=50,
).to(DEVICE)


# model = LitSuperResNet.load_from_checkpoint(f"../model/checkpoints/{COMMENT}/upscaler-epoch=33-latest.ckpt").to(DEVICE)

Model summary output helps in debugging model layer dimensions

In [ ]:
def print_shape_hook(module, input, output):
    print(f"{module.__class__.__name__} | input: {[i.shape for i in input]} -> output: {output.shape}")

from torchinfo import summary
summary(model, input_size=(22, 3, 128, 128), verbose=2, col_names=("input_size", "output_size"))

# Training

Lightning is used for model training. It helps visualizing training progress on Tensorboard. 

Tensorboard web interface could be run using the following command in terminal:

`tensorboard --logdir ./model/lightning_logs --reload_interval 5 --reload_multifile true --samples_per_plugin=images=1000`

In [ ]:
import lightning as L

from lightning.pytorch.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger


torch.set_float32_matmul_precision('medium')


checkpoint_cb = ModelCheckpoint(
    dirpath=f"../model/checkpoints/{COMMENT}",
    filename="upscaler-{epoch:03d}",
    save_top_k=3,
    monitor="val/loss",
    mode="min"
)

tb_logging_dataset = SuperResDataset(
    os.getenv("TB_LOGGING_DATA_PATH"), 
    patch_size=HIRES_PATCH_SIZE,
    downscale=(5616 // 2, 3744 // 2), # consistent with previous tensorboard logs
    seed=42,
)

dl_for_logging = DataLoader(tb_logging_dataset, batch_size=8, shuffle=False)
logger_cb = ImageLoggerCallback(dl_for_logging, log_every_n_epochs=1)

trainer = L.Trainer(
    max_epochs=1500,
    limit_train_batches=100,
    limit_val_batches=50,
    logger = TensorBoardLogger("../model/lightning_logs", name=COMMENT),
    log_every_n_steps=5,
    callbacks=[checkpoint_cb, logger_cb],
)

train_dl = DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_dl   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
trainer.fit(model, train_dl, val_dl)

Training process could be interrupted at any time for some reasons. In addition to best checkpoints saving here we could also save the last epoch checkpoint on which training was aborted.

In [ ]:
model.trainer.save_checkpoint(f"../model/checkpoints/{COMMENT}/upscaler-epoch={model.trainer.current_epoch}-latest.ckpt")

Model state could be found at `../model/checkpoints` upon training completion.